# 2. tokenizer2:

### Learning Objectives
By the end of this lesson, you should be able to:

* Build a production-ready BPE Tokenizer that correctly handles Unicode, whitespace normalization, and special tokens.
* Implement byte-level fallback so the Tokenizer can encode any input, including emojis, CJK text, and code, without generating unknown tokens.
* Use a pre-tokenization regex to split text at appropriate word, number, punctuation, and whitespace boundaries before executing BPE merges.
* Train a custom Tokenizer on a corpus and compare its compression ratio on multilingual text with `tiktoken`.
* Understand the role of Chat Templates in converting structured messages into Token IDs.
* Explain the differences between an educational Python implementation and a production-ready Tokenizer in terms of speed, accuracy, and reproducibility.
---
### What is the Problem?

The BPE Tokenizer from Lesson 01 worked on English text. Now test that same Tokenizer with Japanese text, emojis, or Python code containing a mix of tabs and spaces; part of the process will likely break or produce an unsuitable output.

The problem is not with the BPE algorithm itself; the problem is that the implementation is not yet complete. A production-ready Tokenizer must:

- Handle input at the byte level, independent of language;
- Normalize Unicode according to a defined policy before splitting text;
- Have special tokens that are never split or merged with other tokens;
- Combine pre-tokenization with subword splitting;
- Provide reliable, and ideally reversible, encode and decode capabilities;
- Be fast enough not to become a bottleneck in the training pipeline;
- Save vocabulary, merge rules, normalization, and special token configurations in a versioned format so results are reproducible.

The GPT-2 vocabulary contains 50,257 tokens, and Llama 3 uses a vocabulary of 128,256 tokens. For GPT-4 family models, tokenizers typically employ vocabularies on the scale of approximately 100,000 tokens; however, the exact number depends on the model and the encoding used.

These numbers do not belong to small toy examples. The merge tables for such vocabularies are trained on massive volumes of data. Beyond BPE itself, components such as normalization, pre-tokenization, special token handling, and chat template formatting separate a tokenizer limited to a "hello world" phrase from one suitable for extensive internet-scale data.

In this lesson, you will build and understand these very components and the logic behind them.

---

### Core Concept: The Full Pipeline

A production-ready Tokenizer is not just a single algorithm; it is a pipeline composed of several stages, each solving a different problem.

    A[Raw Text] --> B[Normalize] --> C[Pre-tokenize]--> D[BPE Merge]--> E[Special Tokens]--> F[Token IDs]





Packages

In [4]:
import re
import unicodedata
from collections import Counter
from typing import Dict, List, Tuple, Union

In [5]:
import regex

PATTERN = regex.compile(
    r"'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+",
    flags=regex.IGNORECASE,
)

def pre_tokenize(text: str) -> list[str]:
    return PATTERN.findall(text)


print(pre_tokenize("I don't code in Python 3.11!"))

['I', ' don', "'t", ' code', ' in', ' Python', ' 3', '.', '11', '!']


This code is responsible for pre-tokenization based on the standard GPT-2 pattern.

This algorithm splits the text into smaller chunks prior to BPE to prevent consecutive words or punctuation marks from merging:


In [6]:
# create pattern
# if have `regex`, use that but have not use `re`
try:
    import regex
    GPT2_PATTERN = regex.compile(
        r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    )
except ImportError:
    GPT2_PATTERN = re.compile(
        r"""'(?:[sdmt]|ll|ve|re)| ?[a-zA-Z]+| ?[0-9]+| ?[^\s\w]+|\s+(?!\S)|\s+"""
    )



## part 1: `def pre_tokenize`

To implement the `pre_tokenize` method, we must use `findall` with `GPT2_PATTERN` to extract all segments matching the regex rules as a list of strings.


In [7]:
# part 1----------------------------------------------
def pre_tokenize(text: str) -> List[str]:
    """
    Split input text into initial word/symbol chunks using the GPT-2 regex pattern.

    Args:
        text (str): Raw input text string to pre-tokenize.

    Returns:
        List[str]: A list of string chunks matched by the pre-tokenization regex.
    """
    # TODO: Apply GPT2_PATTERN regex iterator over text to extract all chunk string matches
    return GPT2_PATTERN.findall(text)


In [8]:
# manual test
print("/1/".center(40, '-'))
sample_1 = "I am Mohsen Mohebbi. I participated in the Daneshkar Artificial Intelligence course."
print(f"split form:---------------------\n{(sample_1.split())}")
print(f"use function pre_tokenize:------\n{pre_tokenize(sample_1)}")


------------------/1/-------------------
split form:---------------------
['I', 'am', 'Mohsen', 'Mohebbi.', 'I', 'participated', 'in', 'the', 'Daneshkar', 'Artificial', 'Intelligence', 'course.']
use function pre_tokenize:------
['I', ' am', ' Mohsen', ' Mohebbi', '.', ' I', ' participated', ' in', ' the', ' Daneshkar', ' Artificial', ' Intelligence', ' course', '.']


## part 2: `def apply_merge`




In [29]:
# Part 2----------------------------------------------
def apply_merge(byte_seq: List[int], pair: Tuple[int, int], new_id: int) -> List[int]:
    """
    Replace consecutive occurrences of a specific pair of token IDs in a sequence with a new token ID.

    Args:
        byte_seq (List[int]): Current sequence of token IDs.
        pair (Tuple[int, int]): A tuple (first_id, second_id) representing the pair to merge.
        new_id (int): The new token ID assigned to the merged pair.

    Returns:
        List[int]: A new list of token IDs with target pairs merged.
    """
    # TODO: Iterate through byte_seq, find adjacent matching pairs, and replace them with new_id
    # if len byte<2 , NOT merge
    if len(byte_seq) < 2:
        return list(byte_seq)

    # make merge list
    merged: list[int] = []
    i = 0
    first, second = pair

    while i < len(byte_seq):
        # len of text is end? 
        if i < len(byte_seq) - 1 and byte_seq[i] == first and byte_seq[i + 1] == second:
            merged.append(new_id)
            i += 2  # go to next pair
        else:
            merged.append(byte_seq[i])
            i += 1

    return merged


In [30]:
# manual test
print("/1/".center(40, '-'))
sample_1 = "I am Mohsen Mohebbi. I participated in the Daneshkar Artificial Intelligence course."
# use pre_tokenize
test_chunks = pre_tokenize(sample_1)
print(f"main text:-------------------\n{sample_1}")
print(f"first use pre_tokenize:------\n{test_chunks}")

print("/2/".center(40, '-'))
# we need encode
test_bytes = list(sample_1.encode("utf-8"))

if len(test_bytes) < 2:
    print("len byte < 2")


test_pair, test_new_id = (ord("M"), ord("o")), 999

merged: list[int] = []
i = 0
first, second = test_pair

while i < len(test_bytes):
    # len of text is end? 
    if i < len(test_bytes) - 1 and test_bytes[i] == first and test_bytes[i + 1] == second:
        merged.append(test_new_id)
        i += 2  # go to next pair
    else:
        merged.append(test_bytes[i])
        i += 1

print(f"merge is : \n{merged}")

print("/test_fucntion2/".center(40, '-'))
print(f"pair : {test_pair} and new_id : {test_new_id}")

test_apply_merge = apply_merge(test_bytes, test_pair, test_new_id)
print(f"def apply_merge --------------\n{test_apply_merge}")

------------------/1/-------------------
main text:-------------------
I am Mohsen Mohebbi. I participated in the Daneshkar Artificial Intelligence course.
first use pre_tokenize:------
['I', ' am', ' Mohsen', ' Mohebbi', '.', ' I', ' participated', ' in', ' the', ' Daneshkar', ' Artificial', ' Intelligence', ' course', '.']
------------------/2/-------------------
merge is : 
[73, 32, 97, 109, 32, 999, 104, 115, 101, 110, 32, 999, 104, 101, 98, 98, 105, 46, 32, 73, 32, 112, 97, 114, 116, 105, 99, 105, 112, 97, 116, 101, 100, 32, 105, 110, 32, 116, 104, 101, 32, 68, 97, 110, 101, 115, 104, 107, 97, 114, 32, 65, 114, 116, 105, 102, 105, 99, 105, 97, 108, 32, 73, 110, 116, 101, 108, 108, 105, 103, 101, 110, 99, 101, 32, 99, 111, 117, 114, 115, 101, 46]
------------/test_fucntion2/------------
pair : (77, 111) and new_id : 999
def apply_merge --------------
[73, 32, 97, 109, 32, 999, 104, 115, 101, 110, 32, 999, 104, 101, 98, 98, 105, 46, 32, 73, 32, 112, 97, 114, 116, 105, 99, 105, 112, 